In [13]:
%load_ext autoreload
%autoreload 2

import os
import sys

if not os.getcwd().endswith("/quotaclimat"):
    os.chdir("../../../../..")

repo_root_path = os.path.abspath(os.path.dirname(os.getcwd()))
if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)
repo_root_path


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'/root/Workspace'

In [14]:
from quotaclimat.data_ingestion.advertising.s01_detection.e02_create_chunks import (
    ChunkCreatorJob,
    debug_split,
)
from quotaclimat.data_ingestion.advertising.s01_detection.processor import (
    chunk_creator,
)
from quotaclimat.data_ingestion.advertising.tools.segments import Segment
from quotaclimat.data_ingestion.advertising.tools.mediatree.bucket_mediatree import (
    _get_s3_file_basename,
)
from quotaclimat.data_ingestion.advertising.s01_detection.tools.visualizer.split_visualizer import (
    generate_split_visualizer,
)

from datetime import timedelta, datetime


In [15]:
def get_job_from_start(
    channel: str,
    start_sec: float,
    has_previous_segment: bool = False,
    with_next_segment: bool = False,
) -> ChunkCreatorJob:
    """
    Builds a ChunkCreatorJob for the 2-minutes mediatree audio part covering
    `start_sec` (epoch), pointing at the already-downloaded local mp3 (see
    e00_download_audio / notebooks/e01_download_media.ipynb to fetch it first).

    Set `has_previous_segment=True` to mimic a job that is not the first of a
    run (peaks in the first `seconds_reserved_for_previous_segment` seconds are
    dropped). Set `with_next_segment=True` to append the following part's
    margin, exactly like the real pipeline does for non-last jobs.
    """
    start_date = datetime.fromtimestamp(start_sec)
    # floored to the 2min interval mediatree parts are stored at
    rounded_start_date = start_date - timedelta(
        minutes=start_date.minute % 2, seconds=start_date.second, microseconds=start_date.microsecond
    )
    rounded_end_date = rounded_start_date + timedelta(minutes=2)

    segment = Segment(start_date=rounded_start_date, end_date=rounded_end_date, channel=channel)

    audio_file = _get_s3_file_basename(channel, rounded_start_date, rounded_end_date) + ".mp3"
    audio_file_path = "./.cache/mediatree/" + audio_file

    next_audio_file_path = None
    if with_next_segment:
        next_start = rounded_end_date
        next_end = next_start + timedelta(minutes=2)
        next_file = _get_s3_file_basename(channel, next_start, next_end) + ".mp3"
        next_audio_file_path = "./.cache/mediatree/" + next_file

    return ChunkCreatorJob(
        segment=segment,
        audio_file_path=audio_file_path,
        has_previous_segment=has_previous_segment,
        next_audio_file_path=next_audio_file_path,
    )


In [16]:
# Pick two audio windows to compare — e.g. one that splits correctly and one
# you suspect is mis-split (missed cut, or an over-eager cut in the middle of speech).
channel = "tf1"
focus_epoch_a = 1787809779.58
focus_epoch_b = 1787597266.18

job_a = get_job_from_start(channel, focus_epoch_a)
job_b = get_job_from_start(channel, focus_epoch_b)

trace_a = debug_split(job_a, chunk_creator)


DEBUG: audio window splitting analysis
  segment: [2026-08-27 05:48:00 -> 2026-08-27 05:50:00]  channel=tf1
  audio_file_path: ./.cache/mediatree/tf1_2026-08-27T05-48-00Z_2026-08-27T05-50-00Z.mp3
  has_previous_segment=False  next_audio_file_path=None

[1] Load audio: 1920000 samples @ 16000Hz = 120.00s
    total window duration (with margin): 120.00s, 1876 frames

[2] Silence mask (local percentile threshold)
    silence_percentile=5.0  window=±5s
    silent frames: 179/1876 (9.5%)

[3] Peak candidates (deepest point of each silence region)
    35 silence regions found -> 35 candidate peaks
      region[35:43] -> t=2.56s energy=0.00510
      region[48:53] -> t=3.26s energy=0.00444
      region[174:184] -> t=11.39s energy=0.00000
      region[304:308] -> t=19.58s energy=0.00626
      region[310:317] -> t=20.03s energy=0.00557
      region[411:422] -> t=26.82s energy=0.00001
      region[524:531] -> t=33.66s energy=0.00443
      region[568:573] -> t=36.48s energy=0.00291
      region[68

In [17]:
trace_b = debug_split(job_b, chunk_creator)


DEBUG: audio window splitting analysis
  segment: [2026-08-24 18:46:00 -> 2026-08-24 18:48:00]  channel=tf1
  audio_file_path: ./.cache/mediatree/tf1_2026-08-24T18-46-00Z_2026-08-24T18-48-00Z.mp3
  has_previous_segment=False  next_audio_file_path=None

[1] Load audio: 1920000 samples @ 16000Hz = 120.00s
    total window duration (with margin): 120.00s, 1876 frames

[2] Silence mask (local percentile threshold)
    silence_percentile=5.0  window=±5s
    silent frames: 164/1876 (8.7%)

[3] Peak candidates (deepest point of each silence region)
    36 silence regions found -> 36 candidate peaks
      region[19:22] -> t=1.28s energy=0.00758
      region[68:77] -> t=4.74s energy=0.00000
      region[188:193] -> t=12.22s energy=0.01154
      region[285:288] -> t=18.30s energy=0.00298
      region[307:316] -> t=19.97s energy=0.00000
      region[387:391] -> t=24.83s energy=0.00599
      region[412:415] -> t=26.43s energy=0.02048
      region[417:420] -> t=26.75s energy=0.01504
      region[42

In [18]:
html = generate_split_visualizer(
    job_a=job_a,
    job_b=job_b,
    chunk_creator=chunk_creator,
    label_a=f"Window A ({job_a.segment.start_date})",
    label_b=f"Window B ({job_b.segment.start_date})",
    focus_epoch_a=focus_epoch_a,  # zoom in on the exact timestamp you're investigating
    focus_epoch_b=focus_epoch_b,
    zoom_sec=10.0,  # +/- seconds of context around the focus timestamp
)

with open(".cache/split_comparison.html", "w") as f:
    f.write(html)
